# Create Daiwa Anglo-Japanese Foundation Grant Awards (ORG-LEVEL GRANT PATTERN, Method 1 360Giving open data)

The Daiwa Anglo-Japanese Foundation funds UK-Japan collaboration across research, education, arts, culture, and institutional exchange. This ingest covers the foundation's official 360Giving workbooks currently listed in the 360Giving Data Registry: March and September funding rounds from 2021 through 2024.

**Method 1 (direct open-data download).** The 360Giving Data Registry lists publisher `Daiwa Anglo-Japanese Foundation` and resolves to eight direct Excel workbook downloads on `dajf.org.uk`. No browser automation or search API is required.

**Awarding body:** Daiwa Anglo-Japanese Foundation - F4320320179 (GB, ROR 00ev6sa36, DOI 10.13039/501100000594)

**Source:** each workbook has one 360Giving sheet. The source provides stable `Identifier` values, title, description, positive GBP amount, real award date, recipient organization name/identifier/country, and `Grant Programme: Title`.

**Schema choices:**
- One row per grant. `funder_award_id` = source `Identifier`, e.g. `360G-DAJF-14179-15016`.
- `display_name` = source `Title`; `description` = source `Description`.
- `funding_type` = `grant` uniformly; `funder_scheme` = source `Grant Programme: Title` (Daiwa Foundation Small Grant / Daiwa Foundation Award).
- `amount` = positive `Amount Awarded`; `currency` = source `Currency`. Amount coverage is 100%, so runbook section 6.7 is not waived.
- `start_date` = real `Award Date`; `start_year` derived from it. `end_date` / `end_year` are NULL because no grant end date is published.
- `lead_investigator` = org-only lead: given/family/orcid NULL and `affiliation.name` = recipient organization. `affiliation.country` is mapped only from explicit source `Recipient Org:Country` values (`UK` and `Japan`).
- `landing_page_url` is NULL because the workbooks do not expose per-grant pages.

**Contractor handoff:** this notebook is prepared for admin execution after the parquet is uploaded to S3. Contractor has no S3/Databricks access.


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.daiwa_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/daiwa/daiwa_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.daiwa_raw;

## Step 1.5: Inspect raw coverage, countries, programmes, and amounts

Expected after local validation: 232 grants, 2021-2024, 100% title/description/recipient/country/programme/amount/currency/date coverage.

In [ ]:
%sql
DESCRIBE openalex.awards.daiwa_raw;

In [ ]:
%sql
SELECT *
FROM openalex.awards.daiwa_raw
LIMIT 5;

In [ ]:
%sql
-- Uniqueness and high-level coverage from raw staging.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_award_ids,
    COUNT(title) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(award_date) AS has_award_date,
    COUNT(start_year) AS has_start_year,
    COUNT(recipient_org) AS has_recipient_org,
    COUNT(recipient_country_iso) AS has_recipient_country,
    COUNT(grant_programme) AS has_grant_programme,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.daiwa_raw;

In [ ]:
%sql
-- Programme/country/source-round distribution.
SELECT
    grant_programme,
    recipient_country_iso,
    source_round,
    COUNT(*) AS rows,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_gbp
FROM openalex.awards.daiwa_raw
GROUP BY grant_programme, recipient_country_iso, source_round
ORDER BY source_round, grant_programme, recipient_country_iso;

In [ ]:
%sql
-- Top recipient organizations.
SELECT recipient_org, recipient_country_iso, COUNT(*) AS rows,
       ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_gbp
FROM openalex.awards.daiwa_raw
WHERE recipient_org IS NOT NULL
GROUP BY recipient_org, recipient_country_iso
ORDER BY rows DESC, total_gbp DESC
LIMIT 20;

## Step 1.6: Fail-fast - verify Daiwa Anglo-Japanese Foundation funder row exists

Runbook section 2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320320179;

## Step 2: Transform to OpenAlex awards schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.daiwa_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320179  -- Daiwa Anglo-Japanese Foundation
), awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
        ))) % 9000000000 AS id,
        COALESCE(n.title, CONCAT('Daiwa Anglo-Japanese Foundation grant ', n.funder_award_id)) AS display_name,
        n.description,
        f.funder_id,
        n.funder_award_id,
        CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN TRY_CAST(n.amount AS DOUBLE) ELSE NULL END AS amount,
        CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN n.currency ELSE NULL END AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,
        'grant' AS funding_type,
        n.grant_programme AS funder_scheme,
        'daiwa_360giving' AS provenance,
        TRY_CAST(n.award_date AS DATE) AS start_date,
        CAST(NULL AS DATE) AS end_date,
        TRY_CAST(n.start_year AS INT) AS start_year,
        CAST(NULL AS INT) AS end_year,
        CASE
            WHEN n.recipient_org IS NOT NULL THEN struct(
                CAST(NULL AS STRING) AS given_name,
                CAST(NULL AS STRING) AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    n.recipient_org AS name,
                    n.recipient_country_iso AS country,
                    CASE
                        WHEN n.recipient_org_identifier IS NOT NULL THEN array(struct(
                            n.recipient_org_identifier AS id,
                            CAST('360Giving Recipient Org:Identifier' AS STRING) AS type,
                            CAST('source' AS STRING) AS asserted_by
                        ))
                        ELSE CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                    END AS ids
                ) AS affiliation
            )
            ELSE NULL
        END AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        CAST(NULL AS STRING) AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               TRY_CAST(abs(xxhash64(CONCAT(
                   TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
               ))) % 9000000000 AS STRING)) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM openalex.awards.daiwa_raw n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
)
SELECT
    * EXCEPT(start_year, end_year),
    CASE WHEN start_year > YEAR(current_date()) + 1 THEN NULL ELSE start_year END AS start_year,
    CASE WHEN start_year > YEAR(current_date()) + 1 THEN NULL ELSE end_year END AS end_year
FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw at priority 188

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'daiwa_360giving' AND priority = 188;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    188 AS priority  -- Daiwa Anglo-Japanese Foundation grants
FROM openalex.awards.daiwa_awards;

## Step 6: Verification

Amount coverage should be 100% and currency should be GBP. Country is source-authoritative from `Recipient Org:Country` and should split between GB and JP.

In [ ]:
%sql
SELECT COUNT(*) AS total_rows
FROM openalex.awards.daiwa_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.daiwa_awards;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(lead_investigator) AS has_lead,
    COUNT(start_date) AS has_start_date,
    COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.daiwa_awards;

In [ ]:
%sql
-- Runbook section 6.7 amount/currency check. Expected: 232/232 rows, GBP only.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.daiwa_awards;

In [ ]:
%sql
-- Duplicate guard.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.daiwa_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;

In [ ]:
%sql
-- Year sanity. max_start_year should be <= YEAR(current_date()) + 1 after inline cap.
SELECT
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year,
    SUM(CASE WHEN start_year > YEAR(current_date()) + 1 THEN 1 ELSE 0 END) AS future_start_rows
FROM openalex.awards.daiwa_awards;

In [ ]:
%sql
-- Programme split.
SELECT funder_scheme, COUNT(*) AS rows, ROUND(SUM(amount), 0) AS total_gbp
FROM openalex.awards.daiwa_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Country sanity: source-authoritative UK/Japan mapping.
SELECT lead_investigator.affiliation.country AS country, COUNT(*) AS rows
FROM openalex.awards.daiwa_awards
GROUP BY lead_investigator.affiliation.country
ORDER BY rows DESC;

In [ ]:
%sql
-- Funder struct sanity.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.daiwa_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample rows for admin QA.
SELECT
    id,
    SUBSTRING(display_name, 1, 80) AS title,
    funder_scheme,
    funding_type,
    start_date,
    amount,
    currency,
    lead_investigator.affiliation.name AS recipient_org,
    lead_investigator.affiliation.country AS country
FROM openalex.awards.daiwa_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;

## Handoff notes

Contractor-complete status: script and notebook are ready, `CreateAwards.ipynb` has priority 188, and the tracker is marked Step 4 in the PR. Contractor has no S3/Databricks access; admin must upload the parquet, run this notebook, inspect the verification cells, and only then decide whether to wire job YAML.
